In [ ]:
# --- Bootstrap cell : à exécuter en premier sur une VM Colab fraîche ---
# Le noyau Colab tourne sur une machine distante avec son propre système de fichiers,
# séparé de ce notebook ouvert dans VS Code. Il ne voit ni ce fichier ni le dépôt qui
# le contient tant qu'on ne les récupère pas explicitement. Le dépôt étant privé, on
# télécharge uniquement le CSV dont ce notebook a besoin (pas tout le dépôt) via
# l'API GitHub authentifiée par token — pas besoin de "Mount Google Drive", ça n'a
# rien à voir avec l'accès au dépôt.
import os
from getpass import getpass

os.makedirs("data", exist_ok=True)
csv_path = "data/sim_category_daily.csv"

if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
    print(f"Déjà présent sur cette VM : {csv_path} ({os.path.getsize(csv_path)} octets) — rien à faire")
else:
    token = getpass("GitHub token (repo scope, dépôt privé) : ")
    url = "https://raw.githubusercontent.com/youcefsnoussi/dashboard/ai-forecast-gpu-notebook/notebooks/data/sim_category_daily.csv"
    exit_code = os.system(
        f'curl -sfL -H "Authorization: token {token}" "{url}" -o "{csv_path}"'
    )
    del token
    if exit_code != 0 or not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0:
        raise RuntimeError(
            "Échec du téléchargement — vérifie que le token a le scope 'repo' et que "
            "la branche/le chemin ci-dessus sont corrects."
        )
    print(f"Téléchargé : {csv_path} ({os.path.getsize(csv_path)} octets)")


# Prévision de la demande — recherche d'hyperparamètres sur GPU

Ce notebook reprend exactement le pipeline déployé dans l'application (même feature
engineering, même découpage temporel, même donnée réelle — `commercial.facture_detail`
de Groupe SIM, agrégée par catégorie et par jour) et va plus loin : une vraie recherche
de grille sur GPU avec XGBoost, comparée honnêtement contre le modèle actuellement en
production (Random Forest, non réglé) et contre deux références naïves.

**Sur Colab : Exécution → Modifier le type d'exécution → GPU** avant de lancer quoi que
ce soit ci-dessous.

**Honnêteté sur la taille des données avant de commencer :** ce CSV fait ~10 500 lignes
(agrégat quotidien par catégorie, pas les transactions individuelles). Un GPU n'apporte
aucun gain de vitesse mesurable sur un jeu de cette taille — XGBoost tournerait presque
aussi vite sur CPU. Ce notebook est câblé pour le GPU parce que la prochaine étape
naturelle est de refaire cette recherche sur les données ligne par ligne
(`facture_detail` brut, potentiellement des millions de lignes), là où le GPU devient
réellement nécessaire. Le résultat de la recherche de grille lui-même reste valable
indépendamment du matériel utilisé.


In [ ]:
!nvidia-smi || echo "Pas de GPU détecté dans cette session Colab — Exécution > Modifier le type d'exécution > GPU"


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "xgboost", "scikit-learn"], check=True)

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import matplotlib.pyplot as plt

print("xgboost", xgb.__version__)

# get_params() only echoes back what you pass in - it does NOT prove the GPU was
# actually used. "tree_method=gpu_hist" is a deprecated alias on recent XGBoost and
# can silently fall back to CPU. Real proof: fit a small model with device="cuda"
# and check the booster's own device attribute after training, not before.
_probe = xgb.XGBRegressor(tree_method="hist", device="cuda", n_estimators=50)
_probe.fit(np.random.rand(2000, 8), np.random.rand(2000))
_actual_device = _probe.get_booster().save_config()
import json
_device_used = json.loads(_actual_device)["learner"]["generic_param"]["device"]
print("Device actually used by a trained booster:", _device_used)
assert _device_used.startswith("cuda"), "XGBoost did NOT train on GPU - check Runtime > Change runtime type > GPU"


## 1. Charger la donnée réelle

Même source que le modèle déployé : quantité facturée, sommée par catégorie et par
jour, extraite de `proforma_cmd_bl_fact.facture_detail` ⋈ `facture` ⋈ `article` ⋈
`produit` ⋈ `sous_category_produit` ⋈ `category_produit` sur la copie locale restaurée
de la base `commercial`. Le fichier `data/sim_category_daily.csv` de ce dépôt est cet
export brut, sans aucun retraitement.


In [ ]:
df = pd.read_csv("data/sim_category_daily.csv")
df["jour"] = pd.to_datetime(df["jour"], format="%d/%m/%Y")
df = df.groupby(["category", "jour"], as_index=False)["qty"].sum()

print(df["jour"].min().date(), "->", df["jour"].max().date(), f"({len(df)} lignes catégorie-jour)")
df.head()


## 2. Feature engineering — identique au pipeline déployé

Jours de la semaine, mois, week-end, puis les décalages (`lag_7/14/28`) et moyennes
mobiles (`roll_mean_7/28`) qui donnent au modèle une notion de "ce qui s'est passé
récemment". Tout est décalé (`shift`) pour qu'aucune ligne ne voie sa propre valeur
future — c'est la même discipline que le prototype de prévision original sur données
Kaggle.


In [ ]:
counts = df.groupby("category")["jour"].count()
categories = counts[counts >= 120].index.tolist()
print("Catégories retenues (>=120 jours d'historique) :", categories)

full_range = pd.date_range(df["jour"].min(), df["jour"].max(), freq="D")

frames = []
for cat in categories:
    s = df[df["category"] == cat].set_index("jour")["qty"].reindex(full_range, fill_value=0)
    f = pd.DataFrame({"jour": full_range, "category": cat, "qty": s.values})
    f["dow"] = f["jour"].dt.dayofweek
    f["month"] = f["jour"].dt.month
    f["weekend"] = (f["dow"] >= 5).astype(int)
    f["lag_7"] = f["qty"].shift(7)
    f["lag_14"] = f["qty"].shift(14)
    f["lag_28"] = f["qty"].shift(28)
    f["roll_mean_7"] = f["qty"].shift(1).rolling(7).mean()
    f["roll_mean_28"] = f["qty"].shift(1).rolling(28).mean()
    frames.append(f)

data = pd.concat(frames, ignore_index=True).dropna()
data_encoded = pd.get_dummies(data, columns=["category"], prefix="cat")
feature_cols = [c for c in data_encoded.columns if c not in ("jour", "qty")]
print(f"{len(data_encoded)} lignes après feature engineering, {len(feature_cols)} colonnes de features")


## 3. Découpage temporel — identique au pipeline déployé

Coupure = dernière date observée moins 56 jours (8 semaines). Entraînement sur tout ce
qui précède, test sur tout ce qui suit — jamais l'inverse, jamais mélangé.


In [ ]:
cutoff = data_encoded["jour"].max() - pd.Timedelta(days=56)
train = data_encoded[data_encoded["jour"] <= cutoff].reset_index(drop=True)
test  = data_encoded[data_encoded["jour"] >  cutoff].reset_index(drop=True)

print("Coupure :", cutoff.date())
print(f"Train : {len(train)} lignes  |  Test : {len(test)} lignes (jamais vues à l'entraînement)")

cat_cols = [c for c in data_encoded.columns if c.startswith("cat_")]
test_category = test[cat_cols].idxmax(axis=1).str.replace("cat_", "", regex=False)

def mape(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    mask = actual != 0
    return float((np.abs(pred[mask] - actual[mask]) / actual[mask]).mean() * 100) if mask.any() else float("nan")

def mae(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    return float(np.abs(pred - actual).mean())

def score_by_category(pred_array, label):
    rows = []
    for cat in categories:
        m = test_category == cat
        if m.sum() == 0 or test.loc[m, "qty"].sum() == 0:
            continue
        rows.append({"category": cat, "model": label,
                     "mae": round(mae(test.loc[m, "qty"], pred_array[m.values]), 1),
                     "mape": round(mape(test.loc[m, "qty"], pred_array[m.values]), 1)})
    return pd.DataFrame(rows)


## 4. Références naïves — le plancher à battre

Pas des modèles, des suppositions simples : "comme il y a 7 jours" et "la moyenne des
28 derniers jours". Un modèle qui ne bat pas ça n'a pas gagné le droit d'être plus
compliqué.


In [ ]:
naive_lag7  = test["lag_7"].values
naive_avg28 = test["roll_mean_28"].values

results = pd.concat([
    score_by_category(naive_lag7,  "Naïf (lag-7)"),
    score_by_category(naive_avg28, "Naïf (moyenne 28j)"),
])
results


## 5. Le modèle actuellement en production — Random Forest, non réglé

Mêmes hyperparamètres que `train_sim_forecast.py`, le script qui alimente la page
*Prévision IA* de l'application aujourd'hui. Sert de référence : est-ce que la
recherche de grille apporte un vrai gain, ou est-ce qu'on retunerait juste le même
résultat avec plus d'étapes ?


In [ ]:
rf_deployed = RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=3,
                                     random_state=42, n_jobs=-1)
rf_deployed.fit(train[feature_cols], train["qty"])
pred_rf = np.clip(rf_deployed.predict(test[feature_cols]), 0, None)

results = pd.concat([results, score_by_category(pred_rf, "Random Forest (déployé)")])
results


## 6. Recherche de grille sur GPU — XGBoost

`device="cuda"` (l'API actuelle — `tree_method="gpu_hist"` seul est un alias déprécié
qui peut retomber sur CPU sans prévenir sur les versions récentes) pousse chaque arbre
sur le GPU. La validation croisée utilise `TimeSeriesSplit`, pas un k-fold classique —
un k-fold mélangerait des dates futures dans les plis d'entraînement, exactement la
fuite qu'on a évité en étape 3. Grille : 3 × 4 × 3 × 2 × 2 = **144 combinaisons**,
chacune évaluée sur 3 découpes temporelles successives, soit 432 entraînements XGBoost.
`n_jobs=1` côté `GridSearchCV` est volontaire : c'est le GPU qui doit paralléliser
l'entraînement de chaque arbre, pas le CPU qui lance plusieurs XGBoost en même temps
et se dispute l'unique GPU disponible sur Colab.


In [ ]:
param_grid = {
    "n_estimators":     [200, 400, 800],
    "max_depth":        [4, 6, 8, 10],
    "learning_rate":    [0.01, 0.05, 0.1],
    "subsample":        [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
}

xgb_gpu = xgb.XGBRegressor(tree_method="hist", device="cuda",
                            objective="reg:squarederror", random_state=42, verbosity=0)

tscv = TimeSeriesSplit(n_splits=3)

search = GridSearchCV(xgb_gpu, param_grid, cv=tscv,
                       scoring="neg_mean_absolute_error", n_jobs=1, verbose=1)

search.fit(train[feature_cols], train["qty"])

print("Meilleurs hyperparamètres :", search.best_params_)
print("Meilleur MAE en validation croisée :", round(-search.best_score_, 1))

# Same proof as the probe above, on the actual best model this time - not optional.
_device_used = json.loads(search.best_estimator_.get_booster().save_config())["learner"]["generic_param"]["device"]
print("Device used by the winning model:", _device_used)
assert _device_used.startswith("cuda"), "The grid search itself ran on CPU, not GPU"


In [ ]:
best_xgb = search.best_estimator_
pred_xgb = np.clip(best_xgb.predict(test[feature_cols]), 0, None)

results = pd.concat([results, score_by_category(pred_xgb, "XGBoost (GPU, réglé)")])
results.pivot(index="category", columns="model", values="mape").round(1)


## 7. Le vrai test : est-ce que XGBoost bat le modèle déjà en production ?

Pas "est-ce que le MAPE a l'air bien" — est-ce qu'il bat les deux références naïves
**et** le Random Forest déployé, catégorie par catégorie. Si non sur certaines
catégories, ce tableau le dit aussi, honnêtement.


In [ ]:
pivot_mae = results.pivot(index="category", columns="model", values="mae").round(1)
pivot_mae = pivot_mae[["Naïf (lag-7)", "Naïf (moyenne 28j)", "Random Forest (déployé)", "XGBoost (GPU, réglé)"]]
pivot_mae


In [ ]:
fig, axes = plt.subplots(len(categories), 1, figsize=(11, 2.6 * len(categories)), sharex=False)
for ax, cat in zip(axes, categories):
    m = (test_category == cat).values
    if m.sum() == 0:
        continue
    dates = test.loc[m, "jour"]
    ax.plot(dates, test.loc[m, "qty"], color="#8792a6", lw=1.6, label="Réel")
    ax.plot(dates, pred_xgb[m], color="#2a78d6", lw=1.6, label="Prévu (XGBoost GPU)")
    ax.set_title(cat, loc="left", fontsize=11, fontweight="bold")
    ax.legend(fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("xgboost_actual_vs_predicted.png", dpi=140)
plt.show()


## 8. Sauvegarder — pour rapatrier dans l'application si le gain est réel

Écrit un modèle (`best_xgb_model.json`) et un CSV de prédictions dans le même format
que `ai_forecast.demand_test` déjà utilisé par la page *Prévision IA* — remplaçable
directement si les résultats ci-dessus justifient de changer de modèle en production.


In [ ]:
best_xgb.save_model("best_xgb_model.json")

out = test[["jour"]].copy()
out["category"] = test_category.values
out["actual"] = test["qty"].values
out["predicted"] = pred_xgb
out["date"] = out["jour"].dt.strftime("%Y-%m-%d")
out[["category", "date", "actual", "predicted"]].to_csv("xgboost_predictions.csv", index=False)

print("Écrit : best_xgb_model.json, xgboost_predictions.csv, xgboost_actual_vs_predicted.png")


## Conclusion — à remplir après exécution

Ce notebook calcule tout à l'exécution ; les chiffres exacts dépendent du run. Avant de
proposer de remplacer le Random Forest en production, vérifier explicitement :

- XGBoost bat-il le Random Forest déployé sur **la majorité** des catégories, ou
  seulement une ou deux (auquel cas le gain est peut-être du bruit) ?
- Le meilleur `max_depth`/`n_estimators` trouvé par la grille est-il proche des bords de
  la grille définie à l'étape 6 ? Si oui, la grille était mal centrée — il faut l'élargir
  et relancer, pas conclure.
- Est-ce que le gain (s'il existe) justifie la complexité opérationnelle d'un second
  modèle à maintenir, ou est-ce que le Random Forest actuel — plus simple, déjà en
  production, déjà compris — reste le meilleur choix malgré un MAPE légèrement plus
  élevé ? Un modèle plus précis de 2 points de MAPE mais plus fragile n'est pas
  automatiquement une amélioration.
